# Bismillah clustering 

In [1]:
! pip install kmodes


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from kmodes.kmodes import KModes
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('skincare_preprocessed.csv')

In [3]:
# ── AMBIL DATA SERUM ──────────────────────────────────────────
feature_cols = [
    'Acne_Fighting', 'Acne_Trigger', 'Anti_Aging', 'Brightening',
    'Dark_Spots', 'Drying', 'Eczema', 'Hydrating', 'Irritating',
    'Large_Pores', 'Oily', 'Redness', 'Reduces_Irritation',
    'Rosacea', 'Scar', 'Skin_Texture', 'Worsen_Oily'
]

df_serum = df[df['type'] == 'Serum'][feature_cols].copy()
print("Shape data Serum:", df_serum.shape)
print("\nSample:")
print(df_serum.head(3))

Shape data Serum: (2199, 17)

Sample:
    Acne_Fighting  Acne_Trigger  Anti_Aging  Brightening  Dark_Spots  Drying  \
5               0             0           1            1           1       0   
44              0             0           0            1           0       0   
61              0             1           1            1           0       1   

    Eczema  Hydrating  Irritating  Large_Pores  Oily  Redness  \
5        0          0           1            0     0        1   
44       1          0           0            0     0        0   
61       1          0           1            0     0        0   

    Reduces_Irritation  Rosacea  Scar  Skin_Texture  Worsen_Oily  
5                    0        0     1             0            0  
44                   0        1     0             0            0  
61                   0        1     1             0            0  


In [4]:
# ── CARI K OPTIMAL UNTUK SETIAP ALGORITMA ────────────────────
k_range = range(2, 9)

silhouette_kmodes = []
silhouette_kmeans = []
silhouette_agg = []

for k in k_range:
    # K-Modes
    km = KModes(n_clusters=k, init='Huang', n_init=5, random_state=42)
    labels_km = km.fit_predict(df_serum)
    silhouette_kmodes.append(silhouette_score(df_serum, labels_km, metric='hamming'))

    # K-Means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_kmeans = kmeans.fit_predict(df_serum)
    silhouette_kmeans.append(silhouette_score(df_serum, labels_kmeans))

    # Agglomerative
    agg = AgglomerativeClustering(n_clusters=k, metric='hamming', linkage='average')
    labels_agg = agg.fit_predict(df_serum)
    silhouette_agg.append(silhouette_score(df_serum, labels_agg, metric='hamming'))

    print(f"K={k} | KModes: {silhouette_kmodes[-1]:.4f} | KMeans: {silhouette_kmeans[-1]:.4f} | Agg: {silhouette_agg[-1]:.4f}")

# ── KESIMPULAN ────────────────────────────────────────────────
print("\nK terbaik per algoritma:")
print(f"K-Modes      : K={k_range[silhouette_kmodes.index(max(silhouette_kmodes))]+2} (score: {max(silhouette_kmodes):.4f})")
print(f"K-Means      : K={k_range[silhouette_kmeans.index(max(silhouette_kmeans))]+2} (score: {max(silhouette_kmeans):.4f})")
print(f"Agglomerative: K={k_range[silhouette_agg.index(max(silhouette_agg))]+2} (score: {max(silhouette_agg):.4f})")

K=2 | KModes: 0.3063 | KMeans: 0.1947 | Agg: 0.3036
K=3 | KModes: 0.2829 | KMeans: 0.1866 | Agg: 0.2696
K=4 | KModes: 0.2911 | KMeans: 0.1990 | Agg: 0.2101
K=5 | KModes: 0.3268 | KMeans: 0.2119 | Agg: 0.1557
K=6 | KModes: 0.3080 | KMeans: 0.2218 | Agg: 0.2340
K=7 | KModes: 0.2885 | KMeans: 0.2333 | Agg: 0.2380
K=8 | KModes: 0.3020 | KMeans: 0.2259 | Agg: 0.2206

K terbaik per algoritma:
K-Modes      : K=7 (score: 0.3268)
K-Means      : K=9 (score: 0.2333)
Agglomerative: K=4 (score: 0.3036)
